In [1]:
from argparse import Namespace
import numpy as np
import pandas as pd
import os
from pathlib import Path

DATA_DIR = Path('../data')

def load_file_path(mode, DATA_DIR=DATA_DIR):

    kv1_file_path = {
        'train': [],
        'dev': [],
        'test': []
    }

    if mode == 'explore':   
        for l1 in ['cn', 'es', 'de']:
            train_folder = DATA_DIR / 'train' / l1
            dev_folder = DATA_DIR / 'dev' / l1
            kv1_file_path['train'].append(
                next(iter(train_folder.glob("*.csv")))
                )
            kv1_file_path['dev'].append(
                next(iter(dev_folder.glob("*.csv")))
                )
    
    return kv1_file_path

def load_data_from_path(mode, kv1_file_path):
    if mode == 'explore':
        train_df = pd.concat([pd.read_csv(f) for f in kv1_file_path['train']], axis=0, ignore_index=True)
        dev_df = pd.concat([pd.read_csv(f) for f in kv1_file_path['dev']], axis=0, ignore_index=True)
        df = pd.concat([train_df, dev_df],
        axis = 0,
        keys = ['train', 'dev'],
        names = ['split']
        ).reset_index(level = ['split']).reset_index(drop=True)
    
    return df

In [2]:
kv1_file_path = load_file_path('explore', DATA_DIR)
df = load_data_from_path('explore', kv1_file_path)

In [3]:
df

,split,item_id,L1,en_target_word,en_target_pos,en_target_clue,L1_source_word,L1_context,GLMM_score
0,train,1,cn,span,noun,s___,持续时间段,智能手机使人的注意力集中时间越来越短。,-2.741183
1,train,2,cn,radically,adverb,r________,彻底地，激进地，根本地,我们应当彻底改变只顾经济发展的思路，以保护日渐脆弱的环境。,-1.337120
2,train,3,cn,supermarket,noun,s__________,超市,许多人选择去超市采购水果蔬菜。,2.212272
3,train,4,cn,airplane,noun,a_______,飞机,一架飞机在比利牛斯山坠毁。,0.813175
4,train,5,cn,trying,adjective,t_____,难熬的，考验人耐心的,等待实验结果的过程往往很难熬。,-4.992504
...,...,...,...,...,...,...,...,...,...
20299,dev,6764,de,apart,adverb,a____,auseinander,Wir mussten den Motor auseinandernehmen.,-0.471560
20300,dev,6765,de,picture,noun,p______,Bild,Das Buch enthält viele bunte Bilder.,2.578608
20301,dev,6766,de,tip,verb,t__,"neigen, kippen",In engen Kurven kann der Wagen leicht kippen.,-1.406334
20302,dev,6767,de,nearest,adjective,n______,am nächsten,Die nächste Tankstelle ist neben dem Supermarkt.,0.130288


In [9]:
df[df['en_target_word'] == 'average']

,split,item_id,L1,en_target_word,en_target_pos,en_target_clue,L1_source_word,L1_context,GLMM_score
2934,train,2935,cn,average,adjective,a______,平均的，一般的，普通的,他有着普通孩子的智力。,1.290630
3677,train,3678,cn,average,noun,a______,平均值，平均数,7， 12和20的平均值是13。,1.724753
5374,train,5375,cn,average,verb,a______,算出…的平均数，平均下来是（某数值）,这里的年降雨量平均600毫米。,1.609341
9025,train,2935,es,average,adjective,a______,media,La edad media de mis alumnos es de quince años...,0.912345
9768,train,3678,es,average,noun,a______,promedio,Como promedio la venta de una casa se lleva a ...,1.485299
11465,train,5375,es,average,verb,a______,promediar,Hay que promediar el valor de las ventas de es...,0.681834
15116,train,2935,de,average,adjective,a______,durchschnittlich,Er ist ein durchschnittlicher Sportler.,0.549454
15859,train,3678,de,average,noun,a______,Durchschnitt,Sein Arbeitsergebnis liegt unter dem Durchschn...,0.619731
17556,train,5375,de,average,verb,a______,"im Durchschnitt, durchschnittlich","Das London Eye hat im Durchschnitt über 3,75 M...",0.203641


In [5]:
df.groupby('en_target_word')['L1'].count().sort_values()

en_target_word
plastic      3
playroom     3
playoff      3
playmate     3
playlist     3
            ..
present      9
average      9
light        9
past         9
like        12
Name: L1, Length: 6236, dtype: int64

tuple: (en_lemma, en_pos, L1_context)

### Frequency-based feature

Source 
    - Zipf frequency by [wordfreq library](https://github.com/rspeer/wordfreq/tree/v3.0.2)
    - BNC list look-up by [bnc-lookup library](https://github.com/craigtrim/bnc-lookup)
        - 10 words aren't covered by the BNC list 

Features
    - Zipf frequency of the word
    - A binary indicator whether the word has a zipf score lower than 3, which indicates the word is very rare in the corpus.
    - A binary indicator whether the word is **very common** according to the BNC frequency list. It means that the word should appears in the top 10 frequency buckets. If the word does not exist in BNC list, it is assigned zero.

In [12]:
from wordfreq import zipf_frequency
import bnc_lookup as bnc

def compute_frequency(w):
    """
    zipf_freq: Zipf frequency of a given word based on corpus used by wordfreq
    rare_ind: whether the given word has a zipf frequency lower than 3.
    is_common: whether the given word appears in the top 10 bucket on BNC frequency list, which is considered very common word. Words not on the list are automatically assigned zero. 
    """
    w = w.lower()
    zipf_freq = zipf_frequency(w, lang='en')
    is_rare = int(zipf_freq < 3)
    if bnc.exists(w):
        is_common = int(bnc.bucket(w) <= 10)
    else:
        is_common = 0
    return zipf_freq, is_rare, is_common

### Psycholinguistic features 

MRC Psycholinguistic DB

In [1]:
import pandas as pd

mrc_df = pd.read_csv("hf://datasets/StephanAkkerman/MRC-psycholinguistic-database/mrc_psycholinguistic_database.csv")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
word_arr = df['en_target_word'].unique()

In [20]:
mrc_df[mrc_df['Word'] == 'billionaire']

,Word,Familiarity,Concreteness,Imageability,Age of Acquisition Rating,Meaningfulness: Coloradao Norms,Meaningfulness: Pavio Norms


In [10]:
mrc_df['Word'] = mrc_df['Word'].apply(lambda x: str(x).lower())
cols = ['Word', 'Familiarity','Concreteness', 'Imageability', 'Age of Acquisition Rating', 'Meaningfulness: Coloradao Norms', 'Meaningfulness: Pavio Norms']
mrc_df = mrc_df[cols]

/tmp/ipykernel_4190/741874951.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mrc_df['Word'] = mrc_df['Word'].apply(lambda x: str(x).lower())


In [21]:
len(set(word_arr).difference(set(mrc_df['Word']).intersection(set(word_arr))))

270

270 English words do not appear in the MRC database, which is nontrivial 

In [11]:
mrc_df[mrc_df['Word'].isin(word_arr)].drop_duplicates().reset_index(drop=True)

,Word,Familiarity,Concreteness,Imageability,Age of Acquisition Rating,Meaningfulness: Coloradao Norms,Meaningfulness: Pavio Norms
0,abandon,510,0,395,0,0,0
1,ability,563,273,327,453,0,560
2,able,575,302,284,0,355,0
3,aboard,0,0,0,0,0,0
4,about,593,227,225,0,300,0
...,...,...,...,...,...,...,...
5965,write,560,446,548,0,492,0
5966,writer,0,0,0,0,0,0
5967,writing,630,467,540,256,0,0
5968,written,0,0,0,0,0,0


### Cognate status

A binary indicator of whether the L1 source words is a cognate word to the English target word.

Source:
- [CogNet](https://github.com/kbatsuren/CogNet#)

In [4]:
import pandas as pd
from pathlib import Path

filename = Path("../../CogNet/CogNet-v2.0.tsv")
cog_df = pd.read_csv(filename, 
                    sep = '\t',
                    on_bad_lines='skip')
word_arr = df['en_target_word'].unique()
langs = np.array(['deu', 'spa', 'zho'])
df_1 = cog_df[cog_df['word 1'].isin(word_arr)].reset_index(drop=True)
mask = (df_1['lang 1'] == 'eng') & (df_1['lang 2'].isin(langs))
cognate_sta = df_1[mask].reset_index(drop=True)

In [5]:
word_cog_df = df[['en_target_word', 'L1', 'L1_source_word']]
cognate_sta = cognate_sta.groupby(['word 1', 'lang 2'])['word 2'].agg(lambda x: x).reset_index()
cognate_sta['lang 2'] = cognate_sta['lang 2'].map({
    'spa': 'es',
    'deu': 'de'
})
cognate_sta.columns = *word_cog_df.columns[:2].tolist(), 'cognates'
cognate_merge = pd.merge(word_cog_df, cognate_sta, how = 'left', on = ['en_target_word', 'L1'])

In [35]:
cognate_merge['cognates'].dropna().apply(lambda x: isinstance(x, (str|np.ndarray))))

True

In [40]:
cognate_merge['cognates'].iloc[0] is np.nan

True

In [44]:
import re

def check_match(row):
    # 1. Clean the prompt: lowercase and remove non-alphanumeric (keeping spaces)
    prompt_clean = re.sub(r'[^\w\s]', '', str(row['L1_source_word']).lower())
    prompt_set = set(prompt_clean.split())
    
    # 2. Clean the target: ensure it's a list and lowercase
    targets = row['cognates']
    if isinstance(targets, str):
        targets = [targets]
    
    return any(t.lower() in prompt_set for t in targets) if targets is not np.nan else 0

cognate_merge['is_cognate'] = cognate_merge.apply(check_match, axis=1)

In [46]:
cognate_merge[cognate_merge['is_cognate'] == 1]

,en_target_word,L1,L1_source_word,cognates,is_cognate
6093,supermarket,es,supermercado,supermercado,True
6106,tall,es,alto,alto,True
6114,religious,es,religioso,"[religioso, religioso, religioso]",True
6130,tea,es,té,"[té, té, té, té]",True
6168,kitchen,es,cocina,cocina,True
...,...,...,...,...,...
20277,emotionless,de,"emotionslos, gefühllos",emotionslos,True
20288,cold,de,Kälte,"[kälte, kalt, kalt]",True
20289,weather,de,Wetter,wetter,True
20295,guarantee,de,Garantie,"[garantie, garantie, garantieren, garantieren]",True


### Language Distance

[lang2vec](https://github.com/antonisa/lang2vec)

- genetic distance
- geographic distance
- syntactic distance
- inventory distance 
- phonological distance
- featural distance
- Aligment-based Language vector [Malaviya et al, 2017](https://arxiv.org/pdf/1707.09569)

[CogNet](https://github.com/kbatsuren/CogNet#)
- lexicon similarity



In [5]:
import lang2vec.lang2vec as l2v
from pathlib import Path 

langs = ['eng', 'deu', 'spa', 'zho']
dists = np.stack([l2v.distance(dist, langs)[0, 1:] for dist in l2v.DISTANCES])
de_dist = dists[:, 0]
es_dist = dists[:, 1]
cn_dist = dists[:, 2]

filename = Path('../../lang2vec/lang2vec/data/learned.npy')

learned_database = np.load(filename, encoding="latin1", allow_pickle=True).item()
de_vec = learned_database['deu']
es_vec = learned_database['spa']
cn_vec = learned_database['zho']

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/lang2vec/lang2vec.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
de_vec.shape

(512,)

In [ ]:
de_dist, es_dist, cn_dist

(array([0.4286, 0.1   , 0.42  , 0.4364, 0.3277, 0.4   ]),
 array([0.9   , 0.1   , 0.4   , 0.5481, 0.3433, 0.5   ]),
 array([1.    , 1.    , 0.57  , 0.5983, 0.5687, 0.6   ]))

In [ ]:
import pandas as pd
from pathlib import Path

filename = Path('../../CogNet/similarities_1.0.tsv')
lex_sim_df = pd.read_csv(filename, sep = '\t')

# No Germany
mask = (lex_sim_df['ISO_1'].isin(langs)) & (lex_sim_df['ISO_2'].isin(langs))
lex_sim_df = lex_sim_df[mask].reset_index(drop=True)


idx = np.where((lex_sim_df['ISO_1'] == 'eng') | (lex_sim_df['ISO_2'] == 'eng'))[0]

lex_sim_df.iloc[idx]

,ISO_1,LangName_1,ISO_2,LangName_2,Similarity,Robustness
0,deu,German,eng,English,4.720,High
2,eng,English,spa,Spanish,7.907,High
3,eng,English,zho,Chinese,0.012,High


In [50]:
df

,split,item_id,L1,en_target_word,en_target_pos,en_target_clue,L1_source_word,L1_context,GLMM_score
0,train,1,cn,span,noun,s___,持续时间段,智能手机使人的注意力集中时间越来越短。,-2.741183
1,train,2,cn,radically,adverb,r________,彻底地，激进地，根本地,我们应当彻底改变只顾经济发展的思路，以保护日渐脆弱的环境。,-1.337120
2,train,3,cn,supermarket,noun,s__________,超市,许多人选择去超市采购水果蔬菜。,2.212272
3,train,4,cn,airplane,noun,a_______,飞机,一架飞机在比利牛斯山坠毁。,0.813175
4,train,5,cn,trying,adjective,t_____,难熬的，考验人耐心的,等待实验结果的过程往往很难熬。,-4.992504
...,...,...,...,...,...,...,...,...,...
20299,dev,6764,de,apart,adverb,a____,auseinander,Wir mussten den Motor auseinandernehmen.,-0.471560
20300,dev,6765,de,picture,noun,p______,Bild,Das Buch enthält viele bunte Bilder.,2.578608
20301,dev,6766,de,tip,verb,t__,"neigen, kippen",In engen Kurven kann der Wagen leicht kippen.,-1.406334
20302,dev,6767,de,nearest,adjective,n______,am nächsten,Die nächste Tankstelle ist neben dem Supermarkt.,0.130288


In [ ]:
from dataclasses import dataclass

@dataclass

class EngWord:
    word: str
    pos: str
    zipf_freq: float
    is_rare: int
    is_common: int


class Item:
    item_id: int
    split: str
    L1: str
    context: str
    source_word: str
    target_word: str
    target_pos: str
    target_clue: str
    is_cognate: int
    score: float

class L1:
    code: str
    l2v_embed: np.ndarray
    l2v_dist: np.ndarray
    lex_sim: float




In [1]:
from transformers import AutoConfig, AutoModelForSequenceClassification

# Download configuration from huggingface.co and cache.
config = AutoConfig.from_pretrained("xlm-roberta-large")
config

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}